Load Dataset & Prepare Labels

In [ ]:
focus = article.get("Focus")  # capital F


In [ ]:
import json
from datasets import load_dataset

# -----------------------------
# Input / Output paths
# -----------------------------
input_file = "/content/DhoroniDataset_for_LLM .jsonl"   # <-- removed extra space
output_file = "/content/DhoroniDataset_labeled.jsonl"

# -----------------------------
# Focus-to-causes mapping
# -----------------------------
focus2causes = {
    "Air Pollution": ["vehicle_emissions","industrial_emissions","brick_kilns","open_waste_burning","construction_and_road_dust"],
    "Plastic Pollution": ["plastic_waste_in_water","open_waste_burning","solid_waste_dumping"],
    "Water Pollution": ["industrial_wastewater_discharge","sewage_and_drainage","arsenic_contamination","plastic_waste_in_water"],
    "Global Warming": ["fossil_fuel_burning","deforestation_land_use_change","industrial_emissions","coal_power_plants"],
    "Sound Pollution": ["traffic_noise","industrial_noise","construction_noise","loudspeakers_and_social_noise"],
    "Heavy Metal": ["industrial_emissions","industrial_wastewater_discharge","heavy_metals_contamination"],
    "Climate Change": ["fossil_fuel_burning","deforestation_land_use_change","industrial_emissions","agricultural_burning"],
    "Light Pollution": ["urban_street_lighting","industrial_lighting","advertisement_billboards"],
    "Water Resource Pollution": ["industrial_wastewater_discharge","agrochemical_overuse","sewage_and_drainage"],
    "Sea Level Rising": ["global_warming","glacier_melting","deforestation_land_use_change"],
    "Forestry and Tree Plantation": ["deforestation_land_use_change","illegal_logging","land_use_change"],
    "Soil Pollution": ["agrochemical_overuse","industrial_waste_dumping","landfill_leachate"],
    "Wildlife": ["deforestation_land_use_change","habitat_loss","illegal_poaching"],
    "Rain": ["climate_change","global_warming","deforestation_land_use_change"],
    "Natural Disaster": ["climate_change","sea_level_rising","deforestation_land_use_change"]
}

# Flatten all causes
all_causes = sorted(list({c for causes in focus2causes.values() for c in causes}))
cause2idx = {cause:i for i,cause in enumerate(all_causes)}

# -----------------------------
# Build labeled JSONL
# -----------------------------
with open(input_file, "r", encoding="utf-8") as fin, open(output_file, "w", encoding="utf-8") as fout:
    for line in fin:
        article = json.loads(line)

        # IMPORTANT: use lowercase key "focus"
        focus = article.get("focus")
        causes = focus2causes.get(focus, [])

        # Create multi-hot vector
        label_vector = [0]*len(all_causes)
        for cause in causes:
            idx = cause2idx[cause]
            label_vector[idx] = 1

        # Assign new labels
        article["labels"] = label_vector
        fout.write(json.dumps(article, ensure_ascii=False)+"\n")

print("✅ Fixed JSONL created:", output_file)

# -----------------------------
# Load dataset for training
# -----------------------------
dataset = load_dataset("json", data_files={"train": output_file})
print(dataset)
print("Sample row:", dataset["train"][0])


✅ Fixed JSONL created: /content/DhoroniDataset_labeled.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'focus', 'text', 'labels'],
        num_rows: 2300
    })
})
Sample row: {'id': 1, 'focus': 'Air Pollution', 'text': 'Title: ঢাকায় বায়ু দূষণ রোধে করণীয়\nArticle: ID-1.txt', 'labels': [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]}


**Install & imports**


In [ ]:
# Run in a new Colab cell (or your environment)
!pip install -q transformers datasets accelerate evaluate scikit-learn torch


In [ ]:
# Python imports
import json
import os
from pathlib import Path
from datasets import load_dataset, Dataset
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, hamming_loss, jaccard_score
import torch

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)


**Config / file paths (edit if needed)**

In [ ]:
# ====== Edit paths if your files are placed elsewhere ======
INPUT_FILE = "/content/DhoroniDataset_for_LLM.jsonl"   # your raw JSONL (may already have empty "labels")
OUTPUT_FILE = "/content/DhoroniDataset_labeled.jsonl"  # pipeline will create / overwrite this

# Model + training params
MODEL_NAME = "sagorsarker/bangla-bert-base"   # BanglaBERT model on HF hub
MAX_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
THRESHOLD = 0.5   # for converting probabilities -> binary predictions
RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)


**focus2causes mapping & build cause index**

In [ ]:
focus2causes = {
    "Air Pollution": ["vehicle_emissions","industrial_emissions","brick_kilns","open_waste_burning","construction_and_road_dust"],
    "Plastic Pollution": ["plastic_waste_in_water","open_waste_burning","solid_waste_dumping"],
    "Water Pollution": ["industrial_wastewater_discharge","sewage_and_drainage","arsenic_contamination","plastic_waste_in_water"],
    "Global Warming": ["fossil_fuel_burning","deforestation_land_use_change","industrial_emissions","coal_power_plants"],
    "Sound Pollution": ["traffic_noise","industrial_noise","construction_noise","loudspeakers_and_social_noise"],
    "Heavy Metal": ["industrial_emissions","industrial_wastewater_discharge","heavy_metals_contamination"],
    "Climate Change": ["fossil_fuel_burning","deforestation_land_use_change","industrial_emissions","agricultural_burning"],
    "Light Pollution": ["urban_street_lighting","industrial_lighting","advertisement_billboards"],
    "Water Resource Pollution": ["industrial_wastewater_discharge","agrochemical_overuse","sewage_and_drainage"],
    "Sea Level Rising": ["global_warming","glacier_melting","deforestation_land_use_change"],
    "Forestry and Tree Plantation": ["deforestation_land_use_change","illegal_logging","land_use_change"],
    "Soil Pollution": ["agrochemical_overuse","industrial_waste_dumping","landfill_leachate"],
    "Wildlife": ["deforestation_land_use_change","habitat_loss","illegal_poaching"],
    "Rain": ["climate_change","global_warming","deforestation_land_use_change"],
    "Natural Disaster": ["climate_change","sea_level_rising","deforestation_land_use_change"]
}

# Create the full sorted list of causes and mapping to indices
all_causes = sorted(list({c for causes in focus2causes.values() for c in causes}))
cause2idx = {cause: idx for idx, cause in enumerate(all_causes)}
idx2cause = {v: k for k, v in cause2idx.items()}

print("Number of causes:", len(all_causes))
print("Sample causes:", all_causes[:10])


Number of causes: 33
Sample causes: ['advertisement_billboards', 'agricultural_burning', 'agrochemical_overuse', 'arsenic_contamination', 'brick_kilns', 'climate_change', 'coal_power_plants', 'construction_and_road_dust', 'construction_noise', 'deforestation_land_use_change']


# **Fix / create labeled JSONL (handles focus vs Focus and existing labels) **

In [ ]:
def find_key_case_insensitive(obj, target_keys):
    """Return the first key present in obj whose lowercase is in target_keys (set)."""
    for k in obj.keys():
        if k.lower() in target_keys:
            return k
    return None

# Read input, produce output with multi-hot 'labels'
n_total = 0
n_from_existing = 0
n_from_focus = 0
n_unknown_focus = 0

with open('/content/DhoroniDataset_for_LLM .jsonl', "r", encoding="utf-8") as fin, open(OUTPUT_FILE, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        n_total += 1
        article = json.loads(line)

        # detect focus key (case-insensitive)
        focus_key = find_key_case_insensitive(article, {"focus"})
        focus_val = article.get(focus_key) if focus_key else None

        # detect labels key (case-insensitive)
        labels_key = find_key_case_insensitive(article, {"labels", "label"})
        existing_labels = article.get(labels_key) if labels_key else None

        # If existing multi-hot (non-empty), keep it
        if existing_labels and any(existing_labels):
            # ensure it is list of ints
            label_vector = [int(x) for x in existing_labels]
            n_from_existing += 1
        else:
            # Build multi-hot from focus2causes mapping
            causes = focus2causes.get(focus_val, [])
            if not causes:
                # unknown focus -> zero vector (you may prefer ["Unknown"])
                label_vector = [0] * len(all_causes)
                n_unknown_focus += 1
            else:
                label_vector = [0] * len(all_causes)
                for c in causes:
                    idx = cause2idx[c]
                    label_vector[idx] = 1
                n_from_focus += 1

        # write back: ensure 'labels' key (lowercase)
        article['labels'] = label_vector
        fout.write(json.dumps(article, ensure_ascii=False) + "\n")

print(f"Processed {n_total} rows — from_existing={n_from_existing}, from_focus={n_from_focus}, unknown_focus={n_unknown_focus}")
print("Output written to:", OUTPUT_FILE)

Processed 2300 rows — from_existing=0, from_focus=1685, unknown_focus=615
Output written to: /content/DhoroniDataset_labeled.jsonl


# Load the new labeled JSONL into Hugging Face ***datasets***

In [ ]:
from datasets import load_dataset

output_file = "/content/DhoroniDataset_labeled.jsonl"

# Load the dataset directly (only one split)
dataset = load_dataset("json", data_files=output_file, split="train")

print(dataset)
print("Sample:", dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'focus', 'text', 'labels'],
    num_rows: 2300
})
Sample: {'id': 1, 'focus': 'Air Pollution', 'text': 'Title: ঢাকায় বায়ু দূষণ রোধে করণীয়\nArticle: ID-1.txt', 'labels': [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]}


# **Detect text column & prepare tokenization**

In [ ]:
# helper to pick textual column
def find_text_column(ds):
    # common candidates
    for name in ds.column_names:
        if name.lower() in ("text", "article", "content", "body", "title"):
            return name
    # fallback - pick first string feature
    for name, feat in ds.features.items():
        if feat.dtype == "string":
            return name
    raise ValueError("No text column found. Please ensure your JSONL has a text field (e.g., 'text' or 'article').")

text_col = find_text_column(dataset)
print("Using text column:", text_col)


Using text column: text


# **Prepare model config for multi-label & data collator**

In [ ]:
num_labels = len(all_causes)
id2label = {i: cause for i, cause in idx2cause.items()}
label2id = {v: k for k, v in id2label.items()}

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# **Metrics (multi-label) — uses sigmoid + threshold**

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= THRESHOLD).astype(int)
    true = labels.astype(int)

    # micro / macro F1, precision, recall
    f1_micro = f1_score(true, preds, average="micro", zero_division=0)
    f1_macro = f1_score(true, preds, average="macro", zero_division=0)
    precision_micro = precision_score(true, preds, average="micro", zero_division=0)
    recall_micro = recall_score(true, preds, average="micro", zero_division=0)
    subset_acc = accuracy_score(true, preds)  # exact match on all labels
    jaccard = jaccard_score(true, preds, average="samples", zero_division=0)
    hamming = hamming_loss(true, preds)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "subset_accuracy": subset_acc,
        "jaccard_samples": jaccard,
        "hamming_loss": hamming
    }


# **Save artifacts (model, tokenizer, mapping)**

In [ ]:
OUT_DIR = "./banglabert_multilabel_saved"
os.makedirs(OUT_DIR, exist_ok=True)

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# Save cause2idx mapping
with open(os.path.join(OUT_DIR, "cause2idx.json"), "w", encoding="utf-8") as f:
    json.dump(cause2idx, f, ensure_ascii=False, indent=2)

print("Saved model + tokenizer + cause2idx to", OUT_DIR)


Saved model + tokenizer + cause2idx to ./banglabert_multilabel_saved


In [ ]:
from transformers import pipeline
pipe = pipeline("text-classification", model=OUT_DIR, tokenizer=OUT_DIR, top_k=None, device=0 if torch.cuda.is_available() else -1)

def predict_multi(text):
    # pipeline returns logits for single-label by default; safer to run model/tokenizer manually
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits.cpu().numpy()[0]
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)
    selected = [idx2cause[i] for i,v in enumerate(preds) if v==1]
    return {"probs": probs.tolist(), "predicted_causes": selected}

# Test
print(predict_multi("ঢাকায় বায়ু দূষণ রোধে ট্রাফিক নিয়ন্ত্রণ ও ইঞ্জিনিয়ারিং ব্যবস্থা জরুরি।"))


Device set to use cpu


{'probs': [0.5011287331581116, 0.4549424350261688, 0.5578242540359497, 0.4142642617225647, 0.558971643447876, 0.46455565094947815, 0.6014435887336731, 0.5282367467880249, 0.6186317205429077, 0.5114779472351074, 0.42982181906700134, 0.48292770981788635, 0.5296388864517212, 0.4066712558269501, 0.6031171679496765, 0.5611255168914795, 0.51659095287323, 0.5130203366279602, 0.4320419728755951, 0.4033636152744293, 0.482642263174057, 0.3936183750629425, 0.5641917586326599, 0.5982772707939148, 0.5226396322250366, 0.4856471121311188, 0.5664971470832825, 0.4033138155937195, 0.5095726251602173, 0.5557048320770264, 0.4088257849216461, 0.40927496552467346, 0.42369917035102844], 'predicted_causes': ['advertisement_billboards', 'agrochemical_overuse', 'brick_kilns', 'coal_power_plants', 'construction_and_road_dust', 'construction_noise', 'deforestation_land_use_change', 'global_warming', 'heavy_metals_contamination', 'illegal_logging', 'illegal_poaching', 'industrial_emissions', 'land_use_change', 'la

# **Load labeled JSONL**

In [ ]:
from datasets import load_dataset

output_file = "/content/DhoroniDataset_labeled.jsonl"

# Load the dataset (single split)
dataset = load_dataset("json", data_files=output_file, split="train")

# Split into train/validation (80/20)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
val_dataset   = dataset["test"]

print(train_dataset)
print(val_dataset)


Dataset({
    features: ['id', 'focus', 'text', 'labels'],
    num_rows: 1840
})
Dataset({
    features: ['id', 'focus', 'text', 'labels'],
    num_rows: 460
})


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-3012789700.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Evaluate the model
# **Check performance on the validation set**

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(val_dataset_tokenized)
y_true = np.array(val_dataset_tokenized["labels"])
y_pred = (predictions.predictions >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=all_causes))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


                                 precision    recall  f1-score   support

       advertisement_billboards       0.00      0.00      0.00         3
           agricultural_burning       0.83      0.36      0.50        67
           agrochemical_overuse       0.00      0.00      0.00        24
          arsenic_contamination       0.00      0.00      0.00        39
                    brick_kilns       0.72      0.41      0.53        51
                 climate_change       0.00      0.00      0.00        39
              coal_power_plants       0.00      0.00      0.00        31
     construction_and_road_dust       0.70      0.41      0.52        51
             construction_noise       0.00      0.00      0.00        11
  deforestation_land_use_change       0.83      0.69      0.75       160
            fossil_fuel_burning       0.69      0.55      0.61        98
                glacier_melting       0.00      0.00      0.00         7
                 global_warming       0.00      0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
import datasets

def tokenize_function(examples):
    return tokenizer(examples[text_col], truncation=True, max_length=MAX_LENGTH)

train_dataset_tokenized = train_dataset.map(tokenize_function, batched=True)
val_dataset_tokenized = val_dataset.map(tokenize_function, batched=True)

# Cast labels to float32 for both training and validation datasets
train_dataset_tokenized = train_dataset_tokenized.cast_column("labels", datasets.Sequence(datasets.Value('float32'), length=len(all_causes)))
val_dataset_tokenized = val_dataset_tokenized.cast_column("labels", datasets.Sequence(datasets.Value('float32'), length=len(all_causes)))


print("Tokenized training dataset:", train_dataset_tokenized)
print("Tokenized validation dataset:", val_dataset_tokenized)

Map:   0%|          | 0/1840 [00:00<?, ? examples/s]

Map:   0%|          | 0/460 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1840 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/460 [00:00<?, ? examples/s]

Tokenized training dataset: Dataset({
    features: ['id', 'focus', 'text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1840
})
Tokenized validation dataset: Dataset({
    features: ['id', 'focus', 'text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 460
})


# **TrainingArguments and Model Training

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    seed=RANDOM_SEED,
    dataloader_num_workers=2,
    report_to="none", # Disable reporting to Weights & Biases
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Train the model
trainer.train()

/tmp/ipython-input-4020594657.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Precision Micro,Recall Micro,Subset Accuracy,Jaccard Samples,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,No log,0.223892,0.343651,0.111242,0.759777,0.222041,0.371739,0.157283,0.068445,41.344400,11.126000,1.403000
2,No log,0.197105,0.470897,0.167120,0.658358,0.366531,0.319565,0.250528,0.066469,45.257700,10.164000,1.282000
3,0.234400,0.182758,0.506433,0.196371,0.685237,0.401633,0.350000,0.270129,0.063175,41.062500,11.202000,1.412000
4,0.234400,0.184194,0.527116,0.211246,0.695187,0.424490,0.356522,0.284027,0.061462,45.949400,10.011000,1.262000
5,0.150100,0.184990,0.531112,0.235699,0.664216,0.442449,0.358696,0.292780,0.063043,42.947800,10.711000,1.350000
6,0.150100,0.185935,0.530792,0.235349,0.661389,0.443265,0.363043,0.294335,0.063241,42.704500,10.772000,1.358000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument i

TrainOutput(global_step=1380, training_loss=0.1709175579789756, metrics={'train_runtime': 4504.105, 'train_samples_per_second': 2.451, 'train_steps_per_second': 0.306, 'total_flos': 172023830745984.0, 'train_loss': 0.1709175579789756, 'epoch': 6.0})

# **Load & Use the Model Anywhere**

In [ ]:
from transformers import pipeline
pipe = pipeline("text-classification", model=OUT_DIR, tokenizer=OUT_DIR, top_k=None, device=0 if torch.cuda.is_available() else -1)

def predict_multi(text):
    # pipeline returns logits for single-label by default; safer to run model/tokenizer manually
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits.cpu().numpy()[0]
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)
    selected = [idx2cause[i] for i,v in enumerate(preds) if v==1]
    return {"probs": probs.tolist(), "predicted_causes": selected}

# Test
print(predict_multi("ঢাকায় বায়ু দূষণ রোধে ট্রাফিক নিয়ন্ত্রণ ও ইঞ্জিনিয়ারিং ব্যবস্থা জরুরি।"))

Device set to use cpu


{'probs': [0.027640894055366516, 0.07454279810190201, 0.052379533648490906, 0.07984764873981476, 0.7907236218452454, 0.04675860330462456, 0.05351659283041954, 0.7935965061187744, 0.052575647830963135, 0.0787651315331459, 0.057131510227918625, 0.036290884017944336, 0.040703434497117996, 0.03759773075580597, 0.05314235761761665, 0.050915446132421494, 0.03299897909164429, 0.8221476674079895, 0.048360180109739304, 0.09777291864156723, 0.03863218054175377, 0.07057476043701172, 0.03761427849531174, 0.02664242684841156, 0.06938440352678299, 0.8229305744171143, 0.10627664625644684, 0.04977399855852127, 0.05759120732545853, 0.08631298691034317, 0.07719670981168747, 0.02490418776869774, 0.8199155926704407], 'predicted_causes': ['brick_kilns', 'construction_and_road_dust', 'industrial_emissions', 'open_waste_burning', 'vehicle_emissions']}


# **Save artifacts (model, tokenizer, mapping)**

In [ ]:
OUT_DIR = "./banglabert_multilabel_saved"
os.makedirs(OUT_DIR, exist_ok=True)

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# Save cause2idx mapping
with open(os.path.join(OUT_DIR, "cause2idx.json"), "w", encoding="utf-8") as f:
    json.dump(cause2idx, f, ensure_ascii=False, indent=2)

print("Saved model + tokenizer + cause2idx to", OUT_DIR)

Saved model + tokenizer + cause2idx to ./banglabert_multilabel_saved


# **“Validation dataset formatting**

In [ ]:
import datasets

val_dataset_tokenized = val_dataset_tokenized.with_format("torch", columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'], output_all_columns=True)
val_dataset_tokenized = val_dataset_tokenized.cast_column("labels", datasets.Sequence(datasets.Value('float32'), length=len(all_causes)))

print(val_dataset_tokenized)
print("Sample:", val_dataset_tokenized[0])

Casting the dataset:   0%|          | 0/460 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'focus', 'text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 460
})
Sample: {'labels': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.]), 'input_ids': tensor([    2,    56, 28221,    30, 12765,   410,  9267,    17,  3577,   217,
         3355,  1457,  2180,  2793,  5264,  8489, 27568,    30, 25555,    17,
        13736,   505,   524,    18,    88, 22684,     3]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1]), 'id': 496, 'focus': 'Environmental Pollution', 'text': 'Title: দূষণে দশক-সেরা এ মাসের ১০ দিনের শহরের বাতাস\nArticle: ID-2453.txt'}


In [ ]:
def tokenize_function(examples):
    return tokenizer(examples[text_col], truncation=True, max_length=MAX_LENGTH)

val_dataset_tokenized = val_dataset.map(tokenize_function, batched=True)

print(val_dataset_tokenized)
print("Sample:", val_dataset_tokenized[0])

Map:   0%|          | 0/460 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'focus', 'text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 460
})
Sample: {'id': 496, 'focus': 'Environmental Pollution', 'text': 'Title: দূষণে দশক-সেরা এ মাসের ১০ দিনের শহরের বাতাস\nArticle: ID-2453.txt', 'labels': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'input_ids': [2, 56, 28221, 30, 12765, 410, 9267, 17, 3577, 217, 3355, 1457, 2180, 2793, 5264, 8489, 27568, 30, 25555, 17, 13736, 505, 524, 18, 88, 22684, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
